<a href="https://colab.research.google.com/github/katrinarna/Homeworks/blob/main/SouthBridge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import numpy as np

# Load prices
prices = pd.read_csv("/content/adjusted_close_monthly_2016_2025.csv")
prices["Date"] = pd.to_datetime(prices["Date"])
prices = prices.set_index("Date")

# Asset names
assets = list(prices.columns)
n = len(assets)

# Convert adjusted close prices to simple monthly returns
returns = prices.pct_change().dropna()

# Estimate mean return vector and covariance matrix
mu = (12 * returns.mean()).values
Sigma = (12 * returns.cov()).values   # numpy matrix

# Optional: make Sigma perfectly symmetric
Sigma = 0.5 * (Sigma + Sigma.T)

print("Assets:")
print(assets)

print("\nAnnualized mean returns:")
print(pd.Series(mu, index=assets))

print("\nAnnualized covariance matrix:")
print(pd.DataFrame(Sigma, index=assets, columns=assets))

Assets:
['EWU', 'SPY', 'IEF', 'LQD', 'DBC']

Annualized mean returns:
EWU    0.090939
SPY    0.157208
IEF    0.009685
LQD    0.033703
DBC    0.084490
dtype: float64

Annualized covariance matrix:
          EWU       SPY       IEF       LQD       DBC
EWU  0.026178  0.018610  0.001189  0.007307  0.014953
SPY  0.018610  0.023574  0.001816  0.007992  0.010277
IEF  0.001189  0.001816  0.004384  0.004331 -0.002598
LQD  0.007307  0.007992  0.004331  0.007199  0.001678
DBC  0.014953  0.010277 -0.002598  0.001678  0.026776


In [18]:
gamma = 10.0   # just a trial value for now

m = gp.Model("southbridge_gamma")
m.Params.OutputFlag = 1

# Decision variables: portfolio weights
x = m.addVars(n, lb=0.0, vtype=GRB.CONTINUOUS, name="x")

# Fully invested constraint
m.addConstr(gp.quicksum(x[i] for i in range(n)) == 1.0, name="budget")

# Objective: maximize mu^T x - (gamma/2) x^T Sigma x
linear_part = gp.quicksum(mu[i] * x[i] for i in range(n))
quad_part = gp.quicksum(Sigma[i, j] * x[i] * x[j] for i in range(n) for j in range(n))

m.setObjective(linear_part - 0.5 * gamma * quad_part, GRB.MAXIMIZE)

m.optimize()

# Print results
if m.status == GRB.OPTIMAL:
    x_opt = np.array([x[i].X for i in range(n)])

    print("\nOptimal portfolio weights:")
    for asset, weight in zip(assets, x_opt):
        print(f"{asset}: {weight:.4f}")

    exp_return = float(mu @ x_opt)
    variance = float(x_opt @ Sigma @ x_opt)
    std_dev = variance**0.5

    print(f"\nExpected annual return  = {exp_return:.6f}")
    print(f"Annual variance         = {variance:.6f}")
    print(f"Annual std. dev.        = {std_dev:.6f}")

Set parameter OutputFlag to value 1
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 1 rows, 5 columns and 5 nonzeros (Max)
Model fingerprint: 0x1a903289
Model has 5 linear objective coefficients
Model has 15 quadratic objective terms
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e-02, 2e-01]
  QObjective range [2e-02, 4e-01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]

Presolve time: 0.01s
Presolved: 1 rows, 5 columns, 5 nonzeros
Presolved model has 15 quadratic objective terms
Ordering time: 0.00s

Barrier statistics:
 Free vars  : 4
 AA' NZ     : 1.000e+01
 Factor NZ  : 1.500e+01
 Factor Ops : 5.500e+01 (less than 1 second per iteration)
 Threads    : 1

                  Objective                Residual
Iter       Primal

In [19]:
import numpy as np
import pandas as pd

gamma_grid = [0.1, 0.5, 1, 2, 5, 10, 20, 50, 100]

results = []
all_weights = []

for gamma in gamma_grid:
    m = gp.Model(f"southbridge_gamma_{gamma}")
    m.Params.OutputFlag = 0

    x = m.addVars(n, lb=0.0, vtype=GRB.CONTINUOUS, name="x")

    m.addConstr(gp.quicksum(x[i] for i in range(n)) == 1.0, name="budget")

    linear_part = gp.quicksum(mu[i] * x[i] for i in range(n))
    quad_part = gp.quicksum(Sigma[i, j] * x[i] * x[j] for i in range(n) for j in range(n))

    m.setObjective(linear_part - 0.5 * gamma * quad_part, GRB.MAXIMIZE)
    m.optimize()

    if m.status == GRB.OPTIMAL:
        x_opt = np.array([x[i].X for i in range(n)])
        exp_return = float(mu @ x_opt)
        variance = float(x_opt @ Sigma @ x_opt)
        std_dev = variance**0.5

        results.append([gamma, exp_return, variance, std_dev])
        all_weights.append(x_opt)

results_df = pd.DataFrame(results, columns=["gamma", "expected_return", "variance", "std_dev"])
display(results_df)

,gamma,expected_return,variance,std_dev
0,0.1,0.157208,0.023574,0.153538
1,0.5,0.157208,0.023574,0.153538
2,1.0,0.157208,0.023574,0.153538
3,2.0,0.157208,0.023574,0.153538
4,5.0,0.157208,0.023574,0.153538
5,10.0,0.112264,0.012115,0.110068
6,20.0,0.066862,0.005305,0.072833
7,50.0,0.039620,0.003398,0.058291
8,100.0,0.030540,0.003125,0.055905


In [20]:
weights_df = pd.DataFrame(all_weights, columns=assets)
weights_df.insert(0, "gamma", results_df["gamma"])
display(weights_df)

,gamma,EWU,SPY,IEF,LQD,DBC
0,0.1,5.751750e-11,1.000000,7.375836e-12,6.184293e-12,1.479274e-11
1,0.5,2.936646e-10,1.000000,8.468940e-12,5.476907e-11,2.751627e-10
2,1.0,3.713391e-10,1.000000,1.032283e-10,1.142339e-10,8.433751e-10
3,2.0,3.964638e-09,1.000000,7.818638e-10,1.176938e-09,8.144689e-09
4,5.0,9.117855e-10,1.000000,1.028684e-09,8.071774e-10,4.660282e-07
5,10.0,1.084734e-08,0.629126,2.402903e-01,1.113742e-08,1.305833e-01
6,20.0,1.974307e-08,0.303367,5.305583e-01,7.074067e-09,1.660743e-01
7,50.0,1.803165e-10,0.107912,7.047191e-01,1.996964e-11,1.873690e-01
8,100.0,3.897160e-10,0.042760,7.627727e-01,4.016583e-11,1.944672e-01


In [21]:
final_df = pd.concat([results_df, weights_df.drop(columns="gamma")], axis=1)
display(final_df)

,gamma,expected_return,variance,std_dev,EWU,SPY,IEF,LQD,DBC
0,0.1,0.157208,0.023574,0.153538,5.751750e-11,1.000000,7.375836e-12,6.184293e-12,1.479274e-11
1,0.5,0.157208,0.023574,0.153538,2.936646e-10,1.000000,8.468940e-12,5.476907e-11,2.751627e-10
2,1.0,0.157208,0.023574,0.153538,3.713391e-10,1.000000,1.032283e-10,1.142339e-10,8.433751e-10
3,2.0,0.157208,0.023574,0.153538,3.964638e-09,1.000000,7.818638e-10,1.176938e-09,8.144689e-09
4,5.0,0.157208,0.023574,0.153538,9.117855e-10,1.000000,1.028684e-09,8.071774e-10,4.660282e-07
5,10.0,0.112264,0.012115,0.110068,1.084734e-08,0.629126,2.402903e-01,1.113742e-08,1.305833e-01
6,20.0,0.066862,0.005305,0.072833,1.974307e-08,0.303367,5.305583e-01,7.074067e-09,1.660743e-01
7,50.0,0.039620,0.003398,0.058291,1.803165e-10,0.107912,7.047191e-01,1.996964e-11,1.873690e-01
8,100.0,0.030540,0.003125,0.055905,3.897160e-10,0.042760,7.627727e-01,4.016583e-11,1.944672e-01
